
# Employee Attrition - Feature Engineering Assignment

**Course:** Feature Engineering  
**Dataset:** Synthetic HR employee attrition data  
**Target:** `Attrition` (`0 = Stayed`, `1 = Left`)

## What this notebook covers

- **Part A:** Dataset understanding and data-quality analysis
- **Part B:** Preprocessing and transformation strategy
- **Part C:** Feature construction and feature selection
- **Part D:** PCA assessment and reusable preprocessing pipeline
- **Part E:** Summary, assumptions, and limitations

> I have kept the approach practical rather than applying every possible technique. My main aim is to produce a clean, explainable, and reusable feature set for future modelling.



## 0. Setup

The cell below works in both Google Colab and a local Jupyter environment.

- In Colab, upload `Graded Assignment Dataset.csv` when prompted.
- In a local environment, keep the CSV in the same folder as the notebook.


In [ ]:

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder, OrdinalEncoder, RobustScaler,
    StandardScaler, FunctionTransformer
)
from sklearn.feature_selection import mutual_info_classif
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

RANDOM_STATE = 42

def show_finding(text):
    display(Markdown(f"**Finding:** {text}"))

print("Libraries loaded successfully.")


In [ ]:

# Locate the dataset.
candidate_paths = [
    "/content/Graded Assignment Dataset.csv",   # Google Colab
    "Graded Assignment Dataset.csv",            # Same folder as notebook
    "/mnt/data/Graded Assignment Dataset.csv"   # Current generated environment
]

data_path = next((p for p in candidate_paths if os.path.exists(p)), None)

if data_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        data_path = next(iter(uploaded))
    except Exception as exc:
        raise FileNotFoundError(
            "CSV not found. Please upload 'Graded Assignment Dataset.csv'."
        ) from exc

df = pd.read_csv(data_path)
print(f"Loaded: {data_path}")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
display(df.head())
show_finding(
    f"The data contains {df.shape[0]} employee records and {df.shape[1]} variables, "
    "including the attrition target."
)


# Part A - Understanding the Dataset

## 1. Structure, feature types, and basic checks

In [ ]:

summary = pd.DataFrame({
    "Data_Type": df.dtypes.astype(str),
    "Non_Null": df.notna().sum(),
    "Missing": df.isna().sum(),
    "Unique_Values": df.nunique(dropna=False)
}).sort_values(["Missing", "Unique_Values"], ascending=[False, False])

display(summary)

numeric_cols_raw = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols_raw = df.select_dtypes(exclude=np.number).columns.tolist()

print(f"Numeric columns: {len(numeric_cols_raw)}")
print(f"Categorical columns: {len(categorical_cols_raw)}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate employee IDs: {df['Employee_ID'].duplicated().sum()}")

show_finding(
    "There are no duplicate rows or repeated employee IDs. Employee_ID is an identifier, "
    "not a behavioural feature, so it should not be used for modelling."
)


## 2. Target distribution

In [ ]:

target_counts = df["Attrition"].value_counts().sort_index()
target_pct = df["Attrition"].value_counts(normalize=True).sort_index() * 100

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Stayed (0)", "Left (1)"], target_counts.values)
ax.set_title("Attrition Distribution")
ax.set_ylabel("Number of Employees")

for bar, pct in zip(bars, target_pct.values):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 40,
        f"{pct:.1f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()

display(pd.DataFrame({"Count": target_counts, "Percentage": target_pct.round(2)}))

show_finding(
    f"Attrition is moderately imbalanced: {target_pct.loc[1]:.1f}% of employees left. "
    "A future model should therefore be evaluated using recall, precision, F1-score or PR-AUC, "
    "not accuracy alone."
)


## 3. Missing-value analysis

In [ ]:

missing = (
    df.isna().sum()
      .to_frame("Missing_Count")
      .assign(Missing_Percent=lambda x: x["Missing_Count"] / len(df) * 100)
      .query("Missing_Count > 0")
      .sort_values("Missing_Percent", ascending=False)
)

display(missing)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(missing.index, missing["Missing_Percent"])
ax.set_title("Missing Values by Feature")
ax.set_ylabel("Missing (%)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

show_finding(
    "Only four variables contain missing values, and the highest missingness is 7%. "
    "This is low enough to retain the rows. Median imputation is suitable for numeric fields "
    "because it is less affected by skewness and outliers than mean imputation."
)


## 4. Numerical distributions and skewness

In [ ]:

numeric_features = [
    c for c in df.select_dtypes(include=np.number).columns
    if c != "Attrition"
]

skewness = df[numeric_features].skew().sort_values(ascending=False)
display(skewness.to_frame("Skewness").head(12))

plot_cols = [
    "Monthly_Salary", "Annual_Bonus", "Stock_Option_Value",
    "Overtime_Hours", "Distance_From_Home", "Engagement_Index"
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), plot_cols):
    ax.hist(df[col].dropna(), bins=30, edgecolor="black", linewidth=0.4)
    ax.set_title(col.replace("_", " "))
    ax.set_ylabel("Frequency")

plt.suptitle("Selected Numerical Distributions", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

strongly_skewed = skewness[skewness.abs() > 1].index.tolist()
print("Variables with |skewness| > 1:")
print(strongly_skewed)

show_finding(
    "Compensation, stock option, distance and overtime variables are strongly right-skewed. "
    "I will use log1p transformation for selected non-negative skewed variables. "
    "I will not remove high values automatically because senior employees can genuinely have "
    "much higher salary, bonus or stock options."
)


## 5. Outlier review using the IQR rule

In [ ]:

outlier_rows = []

for col in numeric_features:
    series = df[col].dropna()
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((series < lower) | (series > upper)).sum()
    outlier_rows.append({
        "Feature": col,
        "Outlier_Count": count,
        "Outlier_Percent": count / len(series) * 100,
        "Lower_Bound": lower,
        "Upper_Bound": upper
    })

outlier_summary = (
    pd.DataFrame(outlier_rows)
      .sort_values("Outlier_Percent", ascending=False)
)

display(outlier_summary.head(12))

top_outliers = outlier_summary.head(8).sort_values("Outlier_Percent")
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top_outliers["Feature"], top_outliers["Outlier_Percent"])
ax.set_title("Features with Highest IQR Outlier Percentage")
ax.set_xlabel("Outliers (%)")
plt.tight_layout()
plt.show()

show_finding(
    "The IQR rule flags many observations in naturally skewed variables. "
    "Deleting all such rows would remove legitimate senior or highly compensated employees. "
    "Transformation and robust scaling are safer than blanket row deletion."
)


## 6. Data consistency checks

In [ ]:

consistency_checks = pd.Series({
    "Years in current role > years at company":
        (df["Years_in_Current_Role"] > df["Years_at_Company"]).sum(),
    "Years at company > total work experience":
        (df["Years_at_Company"] > df["Total_Work_Experience"]).sum(),
    "Total experience implies starting work before age 14":
        ((df["Age"] - df["Total_Work_Experience"]) < 14).sum(),
    "Negative monetary values":
        (df[[
            "Monthly_Salary", "Annual_Bonus",
            "Stock_Option_Value", "Monthly_Expenses"
        ]] < 0).any(axis=1).sum()
}, name="Number_of_Rows")

display(consistency_checks.to_frame())

show_finding(
    "Tenure-related fields are internally consistent. Two rows imply an unusually early career start. "
    "Since the values are possible data-entry errors but are not impossible enough to justify deletion, "
    "I keep them and document the issue."
)


## 7. Relationships with attrition

In [ ]:

comparison_features = [
    "Age", "Years_at_Company", "Overtime_Hours",
    "Job_Satisfaction", "Work_Life_Balance",
    "Engagement_Index", "Financial_Stress_Index",
    "Career_Growth_Index", "Monthly_Salary"
]

attrition_means = df.groupby("Attrition")[comparison_features].mean().T
attrition_means.columns = ["Stayed", "Left"]
attrition_means["Relative_Difference_%"] = (
    (attrition_means["Left"] - attrition_means["Stayed"])
    / attrition_means["Stayed"] * 100
)
display(attrition_means.round(2))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(
    axes,
    ["Overtime_Hours", "Work_Life_Balance", "Financial_Stress_Index"]
):
    data_0 = df.loc[df["Attrition"] == 0, col].dropna()
    data_1 = df.loc[df["Attrition"] == 1, col].dropna()
    ax.boxplot([data_0, data_1], labels=["Stayed", "Left"], showfliers=False)
    ax.set_title(col.replace("_", " "))

plt.suptitle("Selected Features by Attrition Group", y=1.04)
plt.tight_layout()
plt.show()

show_finding(
    "Employees who left show higher overtime and financial stress, while their satisfaction, "
    "work-life balance, engagement, career growth and average salary are lower. "
    "These differences provide useful business intuition for constructed features."
)


## 8. Correlation and redundancy

In [ ]:

corr_cols = [
    c for c in numeric_features
    if df[c].nunique() > 2
]
corr = df[corr_cols].corr()

high_corr_pairs = []
for i, col1 in enumerate(corr.columns):
    for col2 in corr.columns[i+1:]:
        value = corr.loc[col1, col2]
        if abs(value) >= 0.80:
            high_corr_pairs.append((col1, col2, value))

high_corr_df = pd.DataFrame(
    high_corr_pairs,
    columns=["Feature_1", "Feature_2", "Correlation"]
).sort_values("Correlation", key=abs, ascending=False)

display(high_corr_df)

selected_corr_cols = [
    "Age", "Total_Work_Experience", "Years_at_Company",
    "Monthly_Salary", "Monthly_Expenses", "Annual_Bonus",
    "Stock_Option_Value", "Annual_Compensation",
    "Performance_Rating", "Manager_Rating",
    "Productivity_Index", "Engagement_Index"
]

fig, ax = plt.subplots(figsize=(10, 8))
matrix = df[selected_corr_cols].corr()
im = ax.imshow(matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(selected_corr_cols)))
ax.set_yticks(range(len(selected_corr_cols)))
ax.set_xticklabels([c.replace("_", " ") for c in selected_corr_cols], rotation=75, ha="right")
ax.set_yticklabels([c.replace("_", " ") for c in selected_corr_cols])
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Correlation Heatmap - Selected Numerical Features")
plt.tight_layout()
plt.show()

show_finding(
    "Annual_Compensation is largely a total of other compensation components, and Experience_Band "
    "is derived from total work experience. These add limited new information. "
    "I will drop them from the final feature set to reduce redundancy and possible multicollinearity."
)



### Part A - Main observations

1. The dataset is clean in terms of duplicates and identifier uniqueness.
2. Attrition is moderately imbalanced, with roughly one employee leaving for every four who stay.
3. Missingness is limited to four numeric variables and remains below 7%.
4. Several compensation and workload variables are strongly right-skewed.
5. High numerical values are not automatically errors; robust treatment is preferred over deletion.
6. Annual compensation and categorical bands duplicate information already present elsewhere.
7. Attrition appears linked with overtime, stress, satisfaction, engagement, career growth and tenure.


# Part B - Designing the Feature Engineering Strategy


## 9. Preprocessing decisions

| Issue | Decision | Reason |
|---|---|---|
| Identifier | Drop `Employee_ID` | Unique label with no generalisable pattern |
| Missing numeric values | Median imputation | Resistant to skew and outliers |
| Nominal categories | One-hot encoding | No artificial order is introduced |
| Ordinal categories | Explicit ordinal encoding | Preserves natural ranking |
| Strong positive skew | `log1p` transformation | Compresses long right tails and supports zero values |
| Different numerical scales | Robust scaling | Less sensitive to extreme values than standard scaling |
| Redundant columns | Drop derived totals/bands | Reduces duplicate information |
| Outliers | Retain, transform and robust-scale | High values may be valid business cases |


# Part C - Feature Construction and Feature Selection

## 10. Construct business-informed features

In [ ]:

def construct_features(data):
    out = data.copy()
    eps = 1e-3

    # Career and tenure
    out["Company_Tenure_Ratio"] = (
        out["Years_at_Company"] /
        (out["Total_Work_Experience"] + 1)
    )
    out["Role_Stability_Ratio"] = (
        out["Years_in_Current_Role"] /
        (out["Years_at_Company"] + 1)
    )
    out["Promotion_Rate"] = (
        out["Promotion_Count"] /
        (out["Years_at_Company"] + 1)
    )

    # Compensation pressure
    out["Expense_to_Salary_Ratio"] = (
        out["Monthly_Expenses"] /
        (out["Monthly_Salary"] + eps)
    )
    out["Bonus_to_Annual_Salary_Ratio"] = (
        out["Annual_Bonus"] /
        (12 * out["Monthly_Salary"] + eps)
    )
    out["Has_Stock_Options"] = (
        out["Stock_Option_Value"] > 0
    ).astype(int)

    # Workload and learning
    out["Overtime_per_Project"] = (
        out["Overtime_Hours"] /
        (out["Project_Count"] + 1)
    )
    out["Learning_Intensity"] = (
        out["Training_Hours"] +
        10 * out["Certification_Count"]
    )

    # Employee experience
    satisfaction_cols = [
        "Job_Satisfaction", "Work_Life_Balance",
        "Environment_Satisfaction",
        "Relationship_Satisfaction"
    ]
    out["Average_Satisfaction"] = out[satisfaction_cols].mean(axis=1)

    out["Support_Recognition_Average"] = (
        out["Manager_Support"] +
        out["Recognition_Score"]
    ) / 2

    out["Career_Engagement_Score"] = (
        out["Career_Growth_Index"] *
        out["Engagement_Index"]
    )

    out["Stress_Satisfaction_Gap"] = (
        out["Financial_Stress_Index"] -
        10 * out["Average_Satisfaction"]
    )

    return out

df_fe = construct_features(df)

engineered_features = [
    "Company_Tenure_Ratio", "Role_Stability_Ratio",
    "Promotion_Rate", "Expense_to_Salary_Ratio",
    "Bonus_to_Annual_Salary_Ratio", "Has_Stock_Options",
    "Overtime_per_Project", "Learning_Intensity",
    "Average_Satisfaction", "Support_Recognition_Average",
    "Career_Engagement_Score", "Stress_Satisfaction_Gap"
]

display(df_fe[engineered_features].head())

show_finding(
    "The new features convert raw HR values into ratios and combined indicators that are easier "
    "to interpret. For example, expense-to-salary ratio represents financial pressure, while "
    "role stability and promotion rate describe career movement relative to tenure."
)


## 11. Quick check of engineered features

In [ ]:

engineered_comparison = (
    df_fe.groupby("Attrition")[engineered_features]
         .mean()
         .T
)
engineered_comparison.columns = ["Stayed", "Left"]
engineered_comparison["Relative_Difference_%"] = (
    (engineered_comparison["Left"] - engineered_comparison["Stayed"])
    / engineered_comparison["Stayed"].replace(0, np.nan) * 100
)

display(
    engineered_comparison
    .sort_values("Relative_Difference_%", key=abs, ascending=False)
    .round(2)
)

plot_engineered = [
    "Expense_to_Salary_Ratio",
    "Average_Satisfaction",
    "Stress_Satisfaction_Gap"
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, plot_engineered):
    means = df_fe.groupby("Attrition")[col].mean()
    ax.bar(["Stayed", "Left"], means.values)
    ax.set_title(col.replace("_", " "))
    ax.set_ylabel("Mean")

plt.suptitle("Engineered Features by Attrition Group", y=1.04)
plt.tight_layout()
plt.show()

show_finding(
    "The engineered features preserve the same business pattern seen in the raw data: "
    "employees who left have lower overall satisfaction and a larger stress-satisfaction gap. "
    "This suggests that the constructed features are meaningful rather than arbitrary combinations."
)


## 12. Final feature set decisions

In [ ]:

target_col = "Attrition"

# Columns excluded deliberately
drop_cols = [
    "Employee_ID",           # identifier
    "Annual_Compensation",   # derived from salary, bonus and stock options
    "Experience_Band",       # derived from total experience
    "Compensation_Band"      # broad version of numerical compensation
]

model_df = df_fe.drop(columns=drop_cols)

print("Dropped columns:", drop_cols)
print("Final raw + engineered feature count:", model_df.shape[1] - 1)

show_finding(
    "I retained Age and Gender in the prepared dataset because they may be useful for bias analysis, "
    "but these protected or sensitive attributes should be reviewed carefully before any real HR "
    "decision system is deployed. A production model may exclude them and use them only for fairness auditing."
)


## 13. Mutual information as a supporting selection check

In [ ]:

# This is an exploratory ranking, not the final preprocessing pipeline.
# Mutual information can capture non-linear dependence with Attrition.

mi_data = model_df.copy()
y_mi = mi_data.pop(target_col)

# Simple temporary encoding only for feature ranking.
for col in mi_data.select_dtypes(include="object").columns:
    mi_data[col] = mi_data[col].astype("category").cat.codes

mi_data = mi_data.apply(
    lambda col: col.fillna(col.median()) if col.dtype != "object" else col
)

mi_scores = mutual_info_classif(
    mi_data,
    y_mi,
    discrete_features="auto",
    random_state=RANDOM_STATE
)

mi_table = (
    pd.DataFrame({
        "Feature": mi_data.columns,
        "Mutual_Information": mi_scores
    })
    .sort_values("Mutual_Information", ascending=False)
)

display(mi_table.head(20))

top_mi = mi_table.head(15).sort_values("Mutual_Information")
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_mi["Feature"], top_mi["Mutual_Information"])
ax.set_title("Top Features by Mutual Information")
ax.set_xlabel("Mutual Information with Attrition")
plt.tight_layout()
plt.show()

show_finding(
    "The ranking is used as supporting evidence rather than an automatic deletion rule. "
    "A low univariate score does not always mean a feature is useless because it may become useful "
    "through interactions with other variables."
)


# Part D - Dimensionality Reduction and Reusable Pipeline

## 14. Define reusable feature construction transformer

In [ ]:

class HRFeatureBuilder(BaseEstimator, TransformerMixin):
    """Add business-informed HR features inside an sklearn pipeline."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return construct_features(X)


class ColumnDropper(BaseEstimator, TransformerMixin):
    """Drop non-predictive or redundant columns."""

    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.columns, errors="ignore")


class DataFramePreprocessor(BaseEstimator, TransformerMixin):
    """Build and fit a ColumnTransformer after feature construction."""

    def __init__(self):
        self.preprocessor_ = None
        self.feature_names_out_ = None

    def fit(self, X, y=None):
        ordinal_cols = [
            "Education_Level",
            "Business_Travel"
        ]

        ordinal_categories = [
            ["High School", "Bachelor", "Master", "Doctorate"],
            ["Never", "Rarely", "Frequently"]
        ]

        nominal_cols = [
            "Gender", "Marital_Status", "Department",
            "Job_Role", "Work_Mode"
        ]

        log_cols = [
            "Years_at_Company", "Years_in_Current_Role",
            "Distance_From_Home", "Monthly_Salary",
            "Annual_Bonus", "Stock_Option_Value",
            "Overtime_Hours", "Monthly_Expenses",
            "Certification_Count", "Promotion_Count",
            "Expense_to_Salary_Ratio",
            "Bonus_to_Annual_Salary_Ratio",
            "Overtime_per_Project"
        ]

        all_numeric = X.select_dtypes(include=np.number).columns.tolist()
        regular_numeric = [
            c for c in all_numeric
            if c not in log_cols and c != "Attrition"
        ]

        log_pipeline = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
            ("scaler", RobustScaler())
        ])

        numeric_pipeline = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler())
        ])

        ordinal_pipeline = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ordinal", OrdinalEncoder(
                categories=ordinal_categories,
                handle_unknown="use_encoded_value",
                unknown_value=-1
            )),
            ("scaler", RobustScaler())
        ])

        nominal_pipeline = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ))
        ])

        self.preprocessor_ = ColumnTransformer(
            transformers=[
                ("log_numeric", log_pipeline, log_cols),
                ("regular_numeric", numeric_pipeline, regular_numeric),
                ("ordinal", ordinal_pipeline, ordinal_cols),
                ("nominal", nominal_pipeline, nominal_cols)
            ],
            remainder="drop",
            verbose_feature_names_out=False
        )

        self.preprocessor_.fit(X, y)
        self.feature_names_out_ = self.preprocessor_.get_feature_names_out()
        return self

    def transform(self, X):
        return self.preprocessor_.transform(X)

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_out_


## 15. Build and run the complete preprocessing pipeline

In [ ]:

X = df.drop(columns=["Attrition"])
y = df["Attrition"]

feature_pipeline = Pipeline(steps=[
    ("feature_builder", HRFeatureBuilder()),
    ("dropper", ColumnDropper(drop_cols)),
    ("preprocessor", DataFramePreprocessor())
])

X_processed = feature_pipeline.fit_transform(X, y)

feature_names = (
    feature_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

X_processed_df = pd.DataFrame(
    X_processed,
    columns=feature_names,
    index=df.index
)

print("Original predictor count:", X.shape[1])
print("Processed feature count:", X_processed_df.shape[1])
print("Missing values after preprocessing:", X_processed_df.isna().sum().sum())
display(X_processed_df.head())

show_finding(
    "The pipeline produces a fully numeric dataset with no missing values. "
    "It also keeps feature construction, imputation, encoding, transformation and scaling together, "
    "which reduces the risk of applying inconsistent steps to future data."
)


## 16. Check effect of log transformation

In [ ]:

before = df["Annual_Bonus"].dropna()
after = np.log1p(before)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(before, bins=35, edgecolor="black", linewidth=0.4)
axes[0].set_title(f"Annual Bonus - Original\nSkewness = {before.skew():.2f}")
axes[1].hist(after, bins=35, edgecolor="black", linewidth=0.4)
axes[1].set_title(f"Annual Bonus - log1p\nSkewness = {after.skew():.2f}")

plt.tight_layout()
plt.show()

show_finding(
    "The log transformation compresses the long upper tail while preserving the ordering of values. "
    "This makes the variable easier for scale-sensitive models to use without discarding high-bonus employees."
)


## 17. PCA assessment

In [ ]:

# PCA requires standardised numerical inputs. The pipeline output is already scaled.
pca = PCA(random_state=RANDOM_STATE)
pca.fit(X_processed_df)

cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
components_90 = np.argmax(cumulative_variance >= 0.90) + 1
components_95 = np.argmax(cumulative_variance >= 0.95) + 1

print(f"Components for 90% variance: {components_90}")
print(f"Components for 95% variance: {components_95}")
print(f"Total processed features: {X_processed_df.shape[1]}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    range(1, len(cumulative_variance) + 1),
    cumulative_variance,
    marker="o",
    markersize=3
)
ax.axhline(0.90, linestyle="--", label="90% variance")
ax.axhline(0.95, linestyle="--", label="95% variance")
ax.axvline(components_95, linestyle=":", label=f"{components_95} components")
ax.set_title("PCA Cumulative Explained Variance")
ax.set_xlabel("Number of Principal Components")
ax.set_ylabel("Cumulative Explained Variance")
ax.legend()
plt.tight_layout()
plt.show()

show_finding(
    f"PCA needs {components_95} components to retain 95% of the variance from "
    f"{X_processed_df.shape[1]} processed features. The reduction is not large enough to justify "
    "the loss of HR interpretability, so PCA is evaluated but not included in the final pipeline."
)



### Why PCA is not used in the final dataset

PCA is technically possible after encoding and scaling, but I decided not to use it as the default output because:

- The processed feature count is still manageable.
- HR teams need to explain which factors influence attrition.
- Principal components mix salary, satisfaction, workload and demographic variables into abstract combinations.
- The reduction required to retain 95% variance is not dramatic.
- PCA may still be useful later for visualisation, clustering, or a model where interpretability is less important.


## 18. Export the final machine-learning-ready dataset

In [ ]:

final_dataset = X_processed_df.copy()
final_dataset["Attrition"] = y.values

output_csv = "employee_attrition_feature_engineered.csv"
final_dataset.to_csv(output_csv, index=False)

print(f"Saved: {output_csv}")
print("Final shape:", final_dataset.shape)
display(final_dataset.head())

try:
    from google.colab import files
    print("In Colab, uncomment the next line to download the file:")
    print("# files.download(output_csv)")
except Exception:
    pass


# Part E - Summary and Reflection


## 19. Final summary

### Important decisions

- Removed the employee identifier because it does not generalise.
- Retained rows with missing values and used median imputation instead of deleting data.
- Applied log transformation to strongly right-skewed, non-negative variables.
- Used robust scaling because several numeric features contain legitimate extreme values.
- One-hot encoded nominal categories and explicitly encoded ordinal categories.
- Removed annual compensation and broad bands where the same information was already available in more detailed fields.
- Constructed career, compensation pressure, workload, learning, satisfaction and stress-related features.
- Evaluated PCA but did not use it in the final output because explainability was more valuable than the limited dimensionality reduction.

### Challenges encountered

- Separating genuine high-value employees from statistical outliers.
- Avoiding duplicate information created by salary bands and compensation totals.
- Balancing predictive usefulness with fairness concerns around age and gender.
- Designing ratios that remain stable when tenure, projects or stock option values are zero.

### Benefits of the final pipeline

- The same transformations can be applied consistently to new employee records.
- Missing-value handling and encoding are contained inside the pipeline, reducing data leakage risk during future model training.
- The output is fully numeric and directly usable by most machine-learning algorithms.
- The features remain understandable to HR stakeholders.

### Assumptions and limitations

- Missing numeric values are assumed to be missing at random.
- The synthetic data may not reproduce all biases or policy constraints of real HR systems.
- No model is trained because the assignment focuses on feature engineering rather than predictive accuracy.
- In a real deployment, fairness, privacy, legal review and human oversight would be mandatory.



## 20. Final checklist

- [x] Dataset structure and quality explored  
- [x] Missing values, skewness, outliers and correlations analysed  
- [x] Feature engineering choices justified  
- [x] Meaningful new features constructed  
- [x] Redundant features reviewed  
- [x] Mutual information used as a supporting selection method  
- [x] PCA assessed and interpreted  
- [x] Reusable preprocessing pipeline created  
- [x] Final machine-learning-ready CSV export included  
